In [ ]:
# !pip install datasets
# !pip install flash-attn --no-build-isolation

In [ ]:
from enum import StrEnum

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

### Configuration

In [ ]:
DATA_NAME = "stsb_multi_mt"
MODEL_NAME = "jinaai/jina-embeddings-v3"
TASK = "text-matching"
BATCH_SIZE = 32
NUM_EPOCHS = 3
LR = 2e-5
MAX_LENGTH = 2048
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Load dataset and create pairs

https://huggingface.co/datasets/mteb/stsb_multi_mt

In [ ]:
sts_dataset = load_dataset(DATA_NAME, name="en", split="train")

In [ ]:
pos_example = sts_dataset[0]
pos_example

In [ ]:
neg_example = sts_dataset[6]
neg_example

In [ ]:
class LabelType(StrEnum):
    POSITIVE = "positive"
    NEGATIVE = "negative"


POSITIVE_THRESHOLD = 4
NEGATIVE_THRESHOLDS = 2

In [ ]:
def create_pair_examples(dataset):
    examples = []
    for item in dataset:
        s1 = item["sentence1"]
        s2 = item["sentence2"]

        if item["similarity_score"] > POSITIVE_THRESHOLD:
            pair_type = LabelType.POSITIVE
        elif item["similarity_score"] < NEGATIVE_THRESHOLDS:
            pair_type = LabelType.NEGATIVE
        else:
            continue
        examples.append([s1, s2, pair_type])

    return examples

In [ ]:
def collate_fn(batch):
    texts1, texts2, labels = zip(*batch)

    inputs1 = model.tokenize(texts1)
    inputs2 = model.tokenize(texts2)
    labels = np.array(labels)

    return {
        "input_ids1": inputs1["input_ids"],
        "attention_mask1": inputs1["attention_mask"],
        "input_ids2": inputs2["input_ids"],
        "attention_mask2": inputs2["attention_mask"],
        "labels": np.where(labels == LabelType.POSITIVE, 1, -1),
    }

In [ ]:
class PairDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

In [ ]:
pair_examples = create_pair_examples(sts_dataset)
pair_dataset = PairDataset(pair_examples)
train_dataloader = DataLoader(
    pair_dataset,
    shuffle=True,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
)

### Load model

- https://arxiv.org/pdf/2409.10173
- https://huggingface.co/jinaai/jina-embeddings-v3

In [ ]:
model = SentenceTransformer(MODEL_NAME, trust_remote_code=True, model_kwargs={"default_task": TASK}).to(DEVICE)

In [ ]:
texts = [
    pos_example["sentence1"],
    pos_example["sentence2"],
    neg_example["sentence1"],
    neg_example["sentence2"],
]
embeddings = model.encode(texts)
embeddings.shape

In [ ]:
similarity_matrix = cosine_similarity(embeddings)
print("Cosine similarity matrix:")
print(similarity_matrix)

### Train model

https://docs.pytorch.org/docs/stable/generated/torch.nn.CosineEmbeddingLoss.html

In [ ]:
class PairLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin
        self.cosine_embedding_loss = nn.CosineEmbeddingLoss(margin=margin)

    def forward(self, embeddings1, embeddings2, labels):
        target = torch.tensor(labels, dtype=torch.float32).to(embeddings1.device)
        return self.cosine_embedding_loss(embeddings1, embeddings2, target)

In [ ]:
loss = PairLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [ ]:
best_val_loss = float("inf")
for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0

    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        encoded_input1 = {
            "input_ids": batch["input_ids1"].to(DEVICE),
            "attention_mask": batch["attention_mask1"].to(DEVICE),
        }
        encoded_input2 = {
            "input_ids": batch["input_ids2"].to(DEVICE),
            "attention_mask": batch["attention_mask2"].to(DEVICE),
        }

        optimizer.zero_grad()

        embeddings1 = model(encoded_input1)["sentence_embedding"]
        embeddings2 = model(encoded_input2)["sentence_embedding"]

        loss = loss(embeddings1, embeddings2, batch["labels"])
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # model.eval()
    # val_loss = 0
    # with torch.no_grad():
    #     for batch in val_dataloader:
    #         encoded_input1 = {
    #             "input_ids": batch["input_ids1"].to(DEVICE),
    #             "attention_mask": batch["attention_mask1"].to(DEVICE),
    #         }
    #         encoded_input2 = {
    #             "input_ids": batch["input_ids2"].to(DEVICE),
    #             "attention_mask": batch["attention_mask2"].to(DEVICE),
    #         }

    #         embeddings1 = model(encoded_input1)["sentence_embedding"]
    #         embeddings2 = model(encoded_input2)["sentence_embedding"]

    #         loss = criterion(embeddings1, embeddings2)
    #         val_loss += loss.item()

    avg_train_loss = train_loss / len(train_dataloader)

    print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}")

print("Training completed!")